# Cat vs. Dog Classifier — Transfer Learning

**What you'll learn:** what pretrained weights are, why freezing saves you, and how a 60-second training run beats a model trained from scratch for hours.

**Estimated time:** 45–60 min &nbsp;|&nbsp; **Runtime:** GPU (Runtime -> Change runtime type -> GPU)

---

### The core idea

A model like ResNet-18 trained on ImageNet has already learned to detect edges, textures, fur patterns, ears, eyes — essentially everything useful for recognizing animals. Instead of learning all that from scratch, we:

1. **Freeze** the pretrained layers (don't change their weights during training)
2. **Replace** only the final classification layer with one that knows about *our* classes
3. **Train just that new layer** — which takes seconds, not hours

? **Think:** Why does this work? What has the pretrained model already learned that we're borrowing?


## 1  Setup

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

if device == 'cpu':
    print("WARNING️  No GPU detected. This will still work but will take longer.")

## 2  Load the data

We use CIFAR-10 and filter to just two classes: cats (label 3) and dogs (label 5).

? **Before running:** What does the `transforms.Normalize` line do, and why do you think we normalize to those specific mean and std values `[0.485, 0.456, 0.406]`?


In [ ]:
# These normalization values come from ImageNet — the dataset ResNet was trained on.
# We normalize our data the same way so the pretrained weights "see" familiar input.
tfm = transforms.Compose([
    transforms.Resize(224),    # ResNet expects 224x224 input
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_train = datasets.CIFAR10('.', train=True,  download=True, transform=tfm)
full_test  = datasets.CIFAR10('.', train=False, download=True, transform=tfm)

# Filter to cats (3) and dogs (5)
# YOUR CODE HERE — complete the list comprehensions
train_idx = [i for i, (_, y) in enumerate(full_train) if y in (3, 5)][:1600]
test_idx  = [i for i, (_, y) in enumerate(full_test)  if y in (3, 5)]

train_dl = DataLoader(Subset(full_train, train_idx), batch_size=32, shuffle=True)
test_dl  = DataLoader(Subset(full_test,  test_idx),  batch_size=64)

print(f"Training images: {len(train_idx)}")
print(f"Test images:     {len(test_idx)}")

Let's visualize a batch so we know what we're working with:


In [ ]:
import numpy as np

# Undo the normalization for display
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

xb, yb = next(iter(train_dl))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
label_map = {3: 'cat', 5: 'dog'}
for i, ax in enumerate(axes.flat):
    img = xb[i].permute(1, 2, 0).numpy()
    img = (img * std + mean).clip(0, 1)
    ax.imshow(img)
    # Labels in Subset are still 3/5 — remap for display
    ax.set_title(label_map.get(int(yb[i]), '?'))
    ax.axis('off')
plt.suptitle('Sample training images', y=1.02)
plt.tight_layout()
plt.show()

## 3  Load the pretrained model and inspect it

In [ ]:
# Load ResNet-18 with ImageNet weights
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Count total parameters before we change anything
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print()
print("Final layer (what we'll replace):")
print(model.fc)

The final layer (`model.fc`) currently outputs **1000 classes** (ImageNet has 1000 categories). We need it to output **2 classes**: cat or dog.

But first: what is `model.fc.in_features`? That's the size of the representation the rest of the network produces — the input to our new classifier.

? **Before running the next cell:** Print `model.fc.in_features`. What is it?


In [ ]:
print("in_features:", model.fc.in_features)
# This is the output size of ResNet-18's feature extractor.
# Our new layer must accept this many inputs.

## 4  Freeze the backbone and replace the head

Now the key step. We want to:
1. Freeze every parameter in the pretrained model (so they don't change during training)
2. Replace `model.fc` with a new `nn.Linear(in_features, 2)` — our 2-class head

Fill in both parts below.


In [ ]:
# Step 1: Freeze all parameters
# YOUR CODE HERE — loop through model.parameters() and set requires_grad = False
for p in model.parameters():
    p.requires_grad = False

# Step 2: Replace the final layer with a 2-class head
# YOUR CODE HERE — model.fc = nn.Linear(???, 2)
model.fc = nn.Linear(model.fc.in_features, 2)

# Move model to GPU/CPU
model = model.to(device)

# Verify
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters: {trainable:,}  ({100 * trainable / (trainable + frozen):.1f}%)")
print(f"Frozen parameters:    {frozen:,}")

? **What fraction of parameters are trainable?** Does that match your prediction from earlier?

We're training less than 0.1% of the model's weights, yet we'll get good accuracy. That's the power of transfer learning.


## 5  Train

We only pass `model.fc.parameters()` to the optimizer — there's no point sending gradients to frozen layers (though PyTorch would just ignore them anyway since `requires_grad=False`).

The `remap` dict converts CIFAR labels (3, 5) to our labels (0, 1) that CrossEntropyLoss expects.


In [ ]:
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
remap = {3: 0, 5: 1}   # cat->0, dog->1

losses = []
model.train()
for epoch in range(3):
    epoch_loss = 0
    for xb, yb in train_dl:
        yb = torch.tensor([remap[int(y)] for y in yb]).to(device)
        xb = xb.to(device)

        # YOUR CODE HERE — complete the training step:
        # 1. Zero gradients
        # 2. Forward pass
        # 3. Compute cross-entropy loss
        # 4. Backward pass
        # 5. Optimizer step
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(train_dl)
    losses.append(avg)
    print(f"Epoch {epoch+1}: avg loss {avg:.3f}")

In [ ]:
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.grid(alpha=0.3)
plt.show()

## 6  Evaluate

? **Predict:** What accuracy do you expect? Random chance is 50% (two classes). A decent classifier should hit... what?


In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_dl:
        yb_mapped = torch.tensor([remap[int(y)] for y in yb]).to(device)
        xb = xb.to(device)
        preds = model(xb).argmax(1)
        correct += (preds == yb_mapped).sum().item()
        total += len(yb_mapped)

print(f"Test accuracy: {correct / total:.3%}  ({correct}/{total} correct)")
print(f"Baseline (random): 50.0%")
print(f"Improvement over random: +{(correct/total - 0.5)*100:.1f}%")

## 7  Look at what it got wrong

In [ ]:
# Find misclassified examples and display them
mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

model.eval()
wrong_imgs, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for xb, yb in test_dl:
        yb_mapped = torch.tensor([remap[int(y)] for y in yb])
        preds = model(xb.to(device)).argmax(1).cpu()
        mask = preds != yb_mapped
        wrong_imgs.extend(xb[mask])
        wrong_true.extend(yb_mapped[mask].tolist())
        wrong_pred.extend(preds[mask].tolist())
        if len(wrong_imgs) >= 12:
            break

names = {0: 'cat', 1: 'dog'}
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    if i >= len(wrong_imgs): break
    img = (wrong_imgs[i].permute(1, 2, 0) * std + mean).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(f"true:{names[wrong_true[i]]}
pred:{names[wrong_pred[i]]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Misclassified examples')
plt.tight_layout()
plt.show()

? **Look at the misclassified images.** Can *you* tell cat from dog easily in these cases? What does that tell you about the model's failure mode?


## 8  Challenge: unfreeze and fine-tune the whole model

So far we only trained the head. Now unfreeze the entire model and train at a much lower learning rate (e.g., `1e-5`). This lets the pretrained weights adapt to our specific task.

```python
for p in model.parameters():
    p.requires_grad = True
opt_full = torch.optim.Adam(model.parameters(), lr=1e-5)
# ... run a few more epochs ...
```

? **Does full fine-tuning beat head-only?** How many more epochs did it take?

This is the real-world decision every transfer-learning practitioner makes.
